In [4]:
# !pip install pandas spacy neo4j networkx scikit-learn
# !python -m spacy download en_core_web_sm
# !pip install --upgrade --force-reinstall numpy==1.26.4 scikit-learn==1.3.2
# !pip install --force-reinstall numpy==1.26.4
# !pip install --force-reinstall scipy==1.10.1
# !pip install --force-reinstall scikit-learn==1.3.2


In [5]:
# !python -m spacy download pt_core_news_lg


In [6]:
import pandas as pd
import spacy
from neo4j import GraphDatabase
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import numpy as np
from collections import Counter, defaultdict

In [7]:
df = pd.read_csv('Ecommerce_KG_Optimized.csv')

In [8]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_description_lenght,product_photos_qty,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,delivery_delay_days,review_length,sentiment_group
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,268.0,4.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,-8,32,Positive
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,178.0,1.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,-6,4,Positive
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,468.0,3.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,-13,20,Positive
3,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22 19:50:18,99.00,30.53,ferramentas_jardim,450.0,1.0,15898b543726a832d4137fbef5d1d00e,1.0,NaN,Aguardando retorno da loja,2017-05-30 00:00:00,-9,4,Negative
4,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,2,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22 19:50:18,99.00,30.53,ferramentas_jardim,450.0,1.0,15898b543726a832d4137fbef5d1d00e,1.0,NaN,Aguardando retorno da loja,2017-05-30 00:00:00,-9,4,Negative


In [9]:
nlp = spacy.load("en_core_web_sm")

doc = nlp(df['review_comment_message'][0])
print(doc)
print(doc.ents)

Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.
(mas ele veio, que ficará chato)


Rows which have entities

In [10]:
# Entities
states_entity = []
reviews_entity = []
sellers_entity = []
products_entity = []
order_items_entity = []
orders_entity = []
customers_entity = []

# Relationships
customer_placed_order = []
order_contains_orderItem = []
orderItem_refersTo_product = []
orderItem_soldBy_seller = []
customer_wrote_review = []
review_reviews_order = []
customer_locatedIn_state = []

for i, row in df.iterrows():
    # if i > 10000:
    #     break
    # Entities
    customers_entity.append({
        'customer_id': row['customer_id'],
        'customer_unique_id': row['customer_unique_id'],
        'customer_city': row['customer_city'],
        'customer_state': row['customer_state']
    })
    orders_entity.append({
        'order_id': row['order_id'],
        'order_status': row['order_status'],
        'order_purchase_timestamp': row['order_purchase_timestamp'],
        'order_approved_at': row['order_approved_at'],
        'order_delivered_carrier_date': row['order_delivered_carrier_date'],
        'order_delivered_customer_date': row['order_delivered_customer_date'],
        'order_estimated_delivery_date': row['order_estimated_delivery_date'],
        'delivery_delay_days': row['delivery_delay_days']
    })
    order_items_entity.append({
        'order_item_id': row['order_item_id'],
        'price': row['price'],
        'freight_value': row['freight_value']
    })
    products_entity.append({
        'product_id': row['product_id'],
        'product_category_name': row['product_category_name'],
        'product_description_lenght': row['product_description_lenght'],
        'product_photos_qty': row['product_photos_qty']
    })
    sellers_entity.append({
        'seller_id': row['seller_id']
    })
    reviews_entity.append({
        'review_id': row['review_id'],
        'review_score': row['review_score'],
        'review_comment_title': row['review_comment_title'],
        'review_comment_message': row['review_comment_message'],
        'review_creation_date': row['review_creation_date'],
        'review_length': row['review_length']
    })
    states_entity.append({
        'name': row['customer_state']
    })

    # Relationships
    customer_placed_order.append(
        [
            {'customer_id': row['customer_id']}, 
            {'order_id': row['order_id']}
        ]
        )
    order_contains_orderItem.append(
        [
            {'order_id': row['order_id']},
            {'order_item_id': row['order_item_id']}  
        ]
    )
    orderItem_refersTo_product.append(
        [
            {'order_item_id': row['order_item_id']}  ,
            {'product_id': row['product_id']}
        ]
    )
    orderItem_soldBy_seller.append(
        [
            {'order_item_id': row['order_item_id']},
            {'seller_id': row['seller_id']}
        ]
    )
    customer_wrote_review.append(
        [
            {'customer_id': row['customer_id']}, 
            {'review_id': row['review_id']}  
        ]
    )
    review_reviews_order.append(
        [
            {'review_id': row['review_id']},
            {'order_id': row['order_id']}
        ]
    )
    customer_locatedIn_state.append(
        [
            {'customer_id': row['customer_id']},
            {'customer_state': row['customer_state']}  
        ]
    )


### Building the Knowledge Graph

In [ ]:
def build_kg_nodes(tx):
    """Create all nodes in the knowledge graph"""
    # Clear existing data
    print("Clearing existing data...")
    tx.run("MATCH (n) DETACH DELETE n")
    
    # Create nodes in batches
    batch_size = 5000
    
    # Create Customer nodes
    if customers_entity:
        print(f"Creating {len(customers_entity)} Customer nodes...")
        for i in range(0, len(customers_entity), batch_size):
            batch = customers_entity[i:i+batch_size]
            tx.run(
                """
                UNWIND $customers AS customer
                MERGE (c:Customer {customer_id: customer.customer_id})
                SET c.customer_unique_id = customer.customer_unique_id,
                    c.customer_city = customer.customer_city,
                    c.customer_state = customer.customer_state
                """,
                customers=batch
            )
    
    # Create Order nodes
    if orders_entity:
        print(f"Creating {len(orders_entity)} Order nodes...")
        for i in range(0, len(orders_entity), batch_size):
            batch = orders_entity[i:i+batch_size]
            tx.run(
                """
                UNWIND $orders AS order
                MERGE (o:Order {order_id: order.order_id})
                SET o.order_status = order.order_status,
                    o.order_purchase_timestamp = order.order_purchase_timestamp,
                    o.order_approved_at = order.order_approved_at,
                    o.order_delivered_carrier_date = order.order_delivered_carrier_date,
                    o.order_delivered_customer_date = order.order_delivered_customer_date,
                    o.order_estimated_delivery_date = order.order_estimated_delivery_date,
                    o.delivery_delay_days = order.delivery_delay_days
                """,
                orders=batch
            )
    
    # Create Order_Item nodes
    if order_items_entity:
        print(f"Creating {len(order_items_entity)} Order_Item nodes...")
        for i in range(0, len(order_items_entity), batch_size):
            batch = order_items_entity[i:i+batch_size]
            tx.run(
                """
                UNWIND $order_items AS order_item
                MERGE (oi:Order_Item {order_item_id: order_item.order_item_id})
                SET oi.price = order_item.price,
                    oi.freight_value = order_item.freight_value
                """,
                order_items=batch
            )
    
    # Create Product nodes (deduplicated)
    print(f"Creating Product nodes...")
    unique_products = []
    seen_ids = set()
    for p in products_entity:
        if p['product_id'] not in seen_ids:
            unique_products.append(p)
            seen_ids.add(p['product_id'])
    
    for i in range(0, len(unique_products), batch_size):
        batch = unique_products[i:i+batch_size]
        tx.run(
            """
            UNWIND $products AS product
            MERGE (p:Product {product_id: product.product_id})
            SET p.product_category_name = product.product_category_name,
                p.product_description_lenght = product.product_description_lenght,
                p.product_photos_qty = product.product_photos_qty
            """,
            products=batch
        )
    
    # Create Seller nodes (deduplicated)
    print(f"Creating Seller nodes...")
    unique_sellers = []
    seen_seller_ids = set()
    for s in sellers_entity:
        if s['seller_id'] not in seen_seller_ids:
            unique_sellers.append(s)
            seen_seller_ids.add(s['seller_id'])
    
    for i in range(0, len(unique_sellers), batch_size):
        batch = unique_sellers[i:i+batch_size]
        tx.run(
            """
            UNWIND $sellers AS seller
            MERGE (s:Seller {seller_id: seller.seller_id})
            """,
            sellers=batch
        )
    
    # Create Review nodes
    if reviews_entity:
        print(f"Creating {len(reviews_entity)} Review nodes...")
        for i in range(0, len(reviews_entity), batch_size):
            batch = reviews_entity[i:i+batch_size]
            tx.run(
                """
                UNWIND $reviews AS review
                MERGE (r:Review {review_id: review.review_id})
                SET r.review_score = review.review_score,
                    r.review_comment_title = review.review_comment_title,
                    r.review_comment_message = review.review_comment_message,
                    r.review_creation_date = review.review_creation_date,
                    r.review_length = review.review_length
                """,
                reviews=batch
            )
    
    # Create State nodes (deduplicated)
    print(f"Creating State nodes...")
    unique_states = []
    seen_states = set()
    for s in states_entity:
        if s['name'] not in seen_states:
            unique_states.append(s)
            seen_states.add(s['name'])
    
    tx.run(
        """
        UNWIND $states AS state
        MERGE (st:State {name: state.name})
        """,
        states=unique_states
    )
    
    return "All nodes created successfully"


def build_kg_relationships(tx):
    """Create all relationships in the knowledge graph"""
    batch_size = 5000
    
    # (Customer) –[:PLACED]→ (Order)
    if customer_placed_order:
        print(f"Creating {len(customer_placed_order)} PLACED relationships...")
        for i in range(0, len(customer_placed_order), batch_size):
            batch = customer_placed_order[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (c:Customer {customer_id: rel[0].customer_id})
                MATCH (o:Order {order_id: rel[1].order_id})
                MERGE (c)-[:PLACED]->(o)
                """,
                relationships=batch
            )
    
    # (Order) –[:CONTAINS]→ (OrderItem)
    if order_contains_orderItem:
        print(f"Creating {len(order_contains_orderItem)} CONTAINS relationships...")
        for i in range(0, len(order_contains_orderItem), batch_size):
            batch = order_contains_orderItem[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (o:Order {order_id: rel[0].order_id})
                MATCH (oi:Order_Item {order_item_id: rel[1].order_item_id})
                MERGE (o)-[:CONTAINS]->(oi)
                """,
                relationships=batch
            )
    
    # (OrderItem) -[:REFERS_TO]→(Product)
    if orderItem_refersTo_product:
        print(f"Creating {len(orderItem_refersTo_product)} REFERS_TO relationships...")
        for i in range(0, len(orderItem_refersTo_product), batch_size):
            batch = orderItem_refersTo_product[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (oi:Order_Item {order_item_id: rel[0].order_item_id})
                MATCH (p:Product {product_id: rel[1].product_id})
                MERGE (oi)-[:REFERS_TO]->(p)
                """,
                relationships=batch
            )
    
    # (OrderItem) –[:SOLD_BY]→ (Seller)
    if orderItem_soldBy_seller:
        print(f"Creating {len(orderItem_soldBy_seller)} SOLD_BY relationships...")
        for i in range(0, len(orderItem_soldBy_seller), batch_size):
            batch = orderItem_soldBy_seller[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (oi:Order_Item {order_item_id: rel[0].order_item_id})
                MATCH (s:Seller {seller_id: rel[1].seller_id})
                MERGE (oi)-[:SOLD_BY]->(s)
                """,
                relationships=batch
            )
    
    # (Customer)–[:WROTE]→ (Review)
    if customer_wrote_review:
        print(f"Creating {len(customer_wrote_review)} WROTE relationships...")
        for i in range(0, len(customer_wrote_review), batch_size):
            batch = customer_wrote_review[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (c:Customer {customer_id: rel[0].customer_id})
                MATCH (r:Review {review_id: rel[1].review_id})
                MERGE (c)-[:WROTE]->(r)
                """,
                relationships=batch
            )
    
    # (Review)–[:REVIEWS]→ (Order)
    if review_reviews_order:
        print(f"Creating {len(review_reviews_order)} REVIEWS relationships...")
        for i in range(0, len(review_reviews_order), batch_size):
            batch = review_reviews_order[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (r:Review {review_id: rel[0].review_id})
                MATCH (o:Order {order_id: rel[1].order_id})
                MERGE (r)-[:REVIEWS]->(o)
                """,
                relationships=batch
            )
    
    # (Customer)-[:LOCATED_IN]→(State)
    if customer_locatedIn_state:
        print(f"Creating {len(customer_locatedIn_state)} LOCATED_IN relationships...")
        for i in range(0, len(customer_locatedIn_state), batch_size):
            batch = customer_locatedIn_state[i:i+batch_size]
            tx.run(
                """
                UNWIND $relationships AS rel
                MATCH (c:Customer {customer_id: rel[0].customer_id})
                MATCH (st:State {name: rel[1].customer_state})
                MERGE (c)-[:LOCATED_IN]->(st)
                """,
                relationships=batch
            )
    
    return "All relationships created successfully"


# Initialize the driver
uri = 'neo4j+s://df998545.databases.neo4j.io'
user = "neo4j"
password = 'P49OO43v3OLd-Gu3PieJtAQw1ifPxlyxU9JUMq7H5MY'

driver = GraphDatabase.driver(uri, auth=(user, password))

# Build KG in two phases
print("=" * 80)
print("BUILDING KNOWLEDGE GRAPH - PHASE 1: NODES")
print("=" * 80)
with driver.session() as session:
    result = session.execute_write(build_kg_nodes)
    print(result)




BUILDING KNOWLEDGE GRAPH - PHASE 1: NODES
Clearing existing data...
Clearing existing data...
Creating 45462 Customer nodes...
Creating 45462 Customer nodes...
Creating 45462 Order nodes...
Creating 45462 Order nodes...
Creating 45462 Order_Item nodes...
Creating 45462 Order_Item nodes...
Creating Product nodes...
Creating Product nodes...
Creating Seller nodes...
Creating Seller nodes...
Creating 45462 Review nodes...
Creating 45462 Review nodes...


In [ ]:
print("\n" + "=" * 80)
print("BUILDING KNOWLEDGE GRAPH - PHASE 2: RELATIONSHIPS")
print("=" * 80)
with driver.session() as session:
    result = session.execute_write(build_kg_relationships)
    print(result)

print("\n" + "=" * 80)
print("Knowledge Graph built successfully!")
print("=" * 80)

In [ ]:
def query_neo4j(query, parameters={}):
    """Execute a Cypher query and return results as a list of dictionaries."""
    with driver.session() as session:
        result = session.run(query, parameters)
        records = [record.data() for record in result]
        return records

In [ ]:
# Query 1: Customers with at least one on-time delivery
query_1 = """
MATCH (c:Customer)-[:PLACED]->(o:Order)
WHERE o.delivery_delay_days <= 0
WITH DISTINCT c
RETURN COUNT(c) AS number_of_customers_with_on_time_delivery
"""

result_1 = query_neo4j(query_1)
print("Query 1 - Customers with at least one on-time delivery:")
print(f"Number of customers: {result_1[0]['number_of_customers_with_on_time_delivery']}")
print(result_1)


Query 1 - Customers with at least one on-time delivery:
Number of customers: 8
[{'number_of_customers_with_on_time_delivery': 8}]


In [ ]:
# Query 2: Top 3 product categories by average review score in a given state
query_2 = """
MATCH (c:Customer {customer_state: $state})-[:PLACED]->(o:Order)<-[:REVIEWS]-(r:Review),
      (o:Order)-[:CONTAINS]->(oi:Order_Item)-[:REFERS_TO]->(p:Product)
WITH p.product_category_name AS category, r.review_score AS score
RETURN category, ROUND(AVG(score), 2) AS avg_review_score
ORDER BY avg_review_score DESC
LIMIT 3
"""

state = "SP"
result_2 = query_neo4j(query_2, {"state": state})
print(f"\nQuery 2 - Top 3 product categories by average review score in {state}:")
for record in result_2:
    print(f"  Category: {record['category']}, Avg Review Score: {record['avg_review_score']}")
print(result_2)



Query 2 - Top 3 product categories by average review score in SP:
  Category: perfumaria, Avg Review Score: 3.75
  Category: pet_shop, Avg Review Score: 3.75
  Category: utilidades_domesticas, Avg Review Score: 3.75
[{'category': 'perfumaria', 'avg_review_score': 3.75}, {'category': 'pet_shop', 'avg_review_score': 3.75}, {'category': 'utilidades_domesticas', 'avg_review_score': 3.75}]


In [ ]:
# Query 3: Number of orders per state (including states with zero orders)
query_3 = """
MATCH (st:State)
OPTIONAL MATCH (c:Customer {customer_state: st.name})-[:PLACED]->(o:Order)
WITH st.name AS state, COUNT(DISTINCT o) AS num_orders
RETURN state, num_orders
ORDER BY state ASC
"""

result_3 = query_neo4j(query_3)
print("\nQuery 3 - Number of orders per state:")
for record in result_3:
    print(f"  State: {record['state']}, Orders: {record['num_orders']}")
print(result_3)



Query 3 - Number of orders per state:
  State: BA, Orders: 2
  State: GO, Orders: 1
  State: RJ, Orders: 2
  State: RN, Orders: 1
  State: SP, Orders: 4
[{'state': 'BA', 'num_orders': 2}, {'state': 'GO', 'num_orders': 1}, {'state': 'RJ', 'num_orders': 2}, {'state': 'RN', 'num_orders': 1}, {'state': 'SP', 'num_orders': 4}]


In [ ]:
# Query 4: Top 3 sellers with lowest average delivery delay
query_4 = """
MATCH (s:Seller)<-[:SOLD_BY]-(oi:Order_Item)<-[:CONTAINS]-(o:Order)
WITH s.seller_id AS seller_id, ROUND(AVG(o.delivery_delay_days), 2) AS avg_delay
RETURN seller_id, avg_delay
ORDER BY avg_delay ASC
LIMIT 3
"""

result_4 = query_neo4j(query_4)
print("\nQuery 4 - Top 3 sellers with lowest average delivery delay:")
for record in result_4:
    print(f"  Seller ID: {record['seller_id']}, Avg Delay (days): {record['avg_delay']}")
print(result_4)



Query 4 - Top 3 sellers with lowest average delivery delay:
  Seller ID: 001cca7ae9ae17fb1caed9dfb1094831, Avg Delay (days): -7.55
  Seller ID: 66922902710d126a0e7d26b0e3805106, Avg Delay (days): -7.4
  Seller ID: 289cdb325fb7e7f891c38608bf9e0962, Avg Delay (days): -7.4
[{'seller_id': '001cca7ae9ae17fb1caed9dfb1094831', 'avg_delay': -7.55}, {'seller_id': '66922902710d126a0e7d26b0e3805106', 'avg_delay': -7.4}, {'seller_id': '289cdb325fb7e7f891c38608bf9e0962', 'avg_delay': -7.4}]


In [ ]:
# Query 5: Products with highest average score
query_5 = """
MATCH (p:Product)<-[:REFERS_TO]-(oi:Order_Item)<-[:CONTAINS]-(o:Order)<-[:REVIEWS]-(r:Review)
WITH p.product_id AS product_id, ROUND(AVG(r.review_score), 2) AS avg_score
WITH ROUND(MAX(avg_score), 2) AS max_avg_score
MATCH (p:Product)<-[:REFERS_TO]-(oi:Order_Item)<-[:CONTAINS]-(o:Order)<-[:REVIEWS]-(r:Review)
WITH p.product_id AS product_id, ROUND(AVG(r.review_score), 2) AS avg_score, max_avg_score
WHERE avg_score = max_avg_score
RETURN product_id, avg_score
ORDER BY product_id
"""

result_5 = query_neo4j(query_5)
print("\nQuery 5 - Products with highest average review score:")
if result_5:
    max_score = result_5[0]['avg_score']
    print(f"  Maximum Average Score: {max_score}")
    for record in result_5:
        print(f"    Product ID: {record['product_id']}, Avg Score: {record['avg_score']}")
else:
    print("  No products found with matching criteria")
print(result_5)



Query 5 - Products with highest average review score:
  Maximum Average Score: 3.3
    Product ID: 009c09f439988bc06a93d6b8186dce73, Avg Score: 3.3
    Product ID: 595fac2a385ac33a80bd5114aec74eb8, Avg Score: 3.3
    Product ID: 5ac9d9e379c606e36a8094a6046f75dc, Avg Score: 3.3
    Product ID: 72d3bf1d3a790f8874096fcf860e3eff, Avg Score: 3.3
    Product ID: 7b717060aa783eb7f23a747a3a733dd7, Avg Score: 3.3
    Product ID: 87285b34884572647811a353c7ac498a, Avg Score: 3.3
    Product ID: d0b61bfb1de832b15ba9d266ca96e5b0, Avg Score: 3.3
    Product ID: d70f38e7f79c630f8ea00c993897042c, Avg Score: 3.3
    Product ID: e99d69efe684efaa643f99805f7c81bc, Avg Score: 3.3
[{'product_id': '009c09f439988bc06a93d6b8186dce73', 'avg_score': 3.3}, {'product_id': '595fac2a385ac33a80bd5114aec74eb8', 'avg_score': 3.3}, {'product_id': '5ac9d9e379c606e36a8094a6046f75dc', 'avg_score': 3.3}, {'product_id': '72d3bf1d3a790f8874096fcf860e3eff', 'avg_score': 3.3}, {'product_id': '7b717060aa783eb7f23a747a3a733dd7',